<a href="https://colab.research.google.com/github/JoseHdez210201/NATS_Connect_IT-OT/blob/main/Masterclass_Nats_IIoT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install nats-py

In [ ]:
import asyncio
import importlib.util
import json
import random
import subprocess
import sys
from datetime import datetime

# 1. Instalar nats-py si no está disponible en el entorno de Colab
if importlib.util.find_spec("nats") is None:
  subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nats-py"])

import nats


async def main(max_mensajes: int | None = 30):
  nc = await nats.connect("nats://demo.nats.io:4222")
  print("🟢 [OT - PLC Simulador] Conectado a NATS. Enviando telemetría...")

  temperatura = 75.0
  nivel = 60.0
  contador = 0

  try:
    while max_mensajes is None or contador < max_mensajes:
      # Simulación de variación de proceso
      temperatura = round(temperatura + random.uniform(-1.5, 2.2), 2)
      if temperatura > 98.0:
        temperatura = 72.0  # Reinicio tras enfriamiento

      nivel = round(
          max(5.0, min(95.0, nivel + random.uniform(-2.0, 2.0))), 1
      )

      payload = {
          "timestamp": datetime.now().strftime("%H:%M:%S"),
          "equipo": "Reactor_01",
          "temperatura_c": temperatura,
          "nivel_pct": nivel,
          "bomba_activa": nivel > 90.0,
      }

      mensaje = json.dumps(payload).encode()
      await nc.publish("planta.linea1.reactor.telemetria_1", mensaje)
      print(f"📤 Enviado ({contador + 1}): {payload}")

      contador += 1
      await asyncio.sleep(1)

  except (asyncio.CancelledError, KeyboardInterrupt):
    print("\n⏹️ Simulación detenida por el usuario.")
  finally:
    await nc.drain()
    await nc.close()
    print("🔴 Conexión NATS cerrada.")


# 2. Ejecutar de forma compatible con el event loop activo de Colab
await main(max_mensajes=30)  # Cambia a None para enviar indefinidamente (usa 'Stop' en Colab)

🟢 [OT - PLC Simulador] Conectado a NATS. Enviando telemetría...
📤 Enviado (1): {'timestamp': '17:02:40', 'equipo': 'Reactor_01', 'temperatura_c': 74.5, 'nivel_pct': 58.2, 'bomba_activa': False}
📤 Enviado (2): {'timestamp': '17:02:41', 'equipo': 'Reactor_01', 'temperatura_c': 76.53, 'nivel_pct': 57.3, 'bomba_activa': False}
📤 Enviado (3): {'timestamp': '17:02:42', 'equipo': 'Reactor_01', 'temperatura_c': 76.48, 'nivel_pct': 59.2, 'bomba_activa': False}
📤 Enviado (4): {'timestamp': '17:02:43', 'equipo': 'Reactor_01', 'temperatura_c': 75.92, 'nivel_pct': 57.8, 'bomba_activa': False}
📤 Enviado (5): {'timestamp': '17:02:44', 'equipo': 'Reactor_01', 'temperatura_c': 77.23, 'nivel_pct': 57.1, 'bomba_activa': False}
📤 Enviado (6): {'timestamp': '17:02:45', 'equipo': 'Reactor_01', 'temperatura_c': 78.58, 'nivel_pct': 56.3, 'bomba_activa': False}
📤 Enviado (7): {'timestamp': '17:02:46', 'equipo': 'Reactor_01', 'temperatura_c': 79.08, 'nivel_pct': 55.5, 'bomba_activa': False}
📤 Enviado (8): {'tim

🔴 Conexión NATS cerrada.
